**Webscraping Companies' Newsrooms**

In [4]:
"""
Newsroom Scraper
========================
Downloads official financial filings for:

  US companies  (Tesla, Disney, Netflix)
    → 10-K  (annual report)      via SEC EDGAR free API
    → DEF 14A (proxy statement)  via SEC EDGAR free API

  German company (Volkswagen)
    → Jahresabschluss / Konzernabschluss
      PRIMARY:  `deutschland` package  (handles Bundesanzeiger CAPTCHA)
      FALLBACK: VW investor-relations page (direct PDF links, no CAPTCHA)

---------------------------------------------------------------------------
INSTALL — run these lines in order to avoid dependency conflicts:

    pip install requests beautifulsoup4
    pip install numpy==1.26.4
    pip install onnxruntime
    pip install deutschland

If `deutschland` still fails (e.g. no onnxruntime wheel for your platform),
the scraper automatically falls back to scraping VW's IR page directly.
No action needed — the fallback is built in.

---------------------------------------------------------------------------
SEC EDGAR requires a User-Agent header that identifies you.
Edit SEC_USER_AGENT below with your name and email before running.

Usage:
    python financial_scraper.py          # all companies, all years
    
    from financial_scraper import Scraper
    Scraper("Tesla", 2023).scrape()
    Scraper("Volkswagen", 2023).scrape()
"""

import os
import re
import time
import logging
from pathlib import Path
from dataclasses import dataclass
from typing import Optional

import requests
from bs4 import BeautifulSoup

# ---------------------------------------------------------------------------
# Config — edit SEC_USER_AGENT before running
# ---------------------------------------------------------------------------

COMPANIES: dict[str, dict] = {
    "Tesla":      {"region": "US", "type": "newsroom"},
    "Disney":     {"region": "US", "type": "newsroom"},
    "Netflix":    {"region": "US", "type": "newsroom"},
    "Volkswagen": {"region": "DE", "type": "newsroom"},
}

YEARS = [2020, 2021, 2022, 2023, 2024, 2025, 2026]

OUTPUT_DIR = Path("newsroom_reports")
REQUEST_DELAY = 3.0   # SEC rate limit is 10 req/s; stay well under

# SEC requires "Name email@example.com" — replace with your own
SEC_USER_AGENT = "Katharina Meder katharina.meder@student.uni-halle.de"

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
)
log = logging.getLogger(__name__)

BROWSER_UA = (
    "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
    "AppleWebKit/537.36 (KHTML, like Gecko) "
    "Chrome/124.0.0.0 Safari/537.36"
)


# ---------------------------------------------------------------------------
# Data model
# ---------------------------------------------------------------------------

@dataclass
class Filing:
    company:    str
    form_type:  str       # "10-K" | "DEF 14A" | "Jahresabschluss" | …
    year:       int
    title:      str
    url:        str
    source:     str       # "SEC_EDGAR" | "Bundesanzeiger" | "VW_IR"
    local_path: Optional[Path] = None
    _content:   str = "" # used for text-only Bundesanzeiger reports


# ---------------------------------------------------------------------------
# Shared helper function
# ---------------------------------------------------------------------------

def _scrape_links(company: str, year: int, page_url: str, source: str) -> list[Filing]:
    """
    Generic link harvester: fetches `page_url`, finds all <a> tags whose
    text or href contains `year`, and returns them as Filing objects.
    PDFs are flagged directly; everything else is treated as HTML.
    """
    headers = {
        "User-Agent": BROWSER_UA,
        "Accept": "text/html,application/xhtml+xml,*/*",
        "Accept-Language": "en-US,en;q=0.9,de;q=0.8",
    }
    try:
        time.sleep(REQUEST_DELAY)
        r = requests.get(page_url, headers=headers, timeout=20, allow_redirects=True)
        r.raise_for_status()
    except requests.RequestException as exc:
        log.warning("%s: fetch failed for %s: %s", source, page_url, exc)
        return []

    soup = BeautifulSoup(r.text, "html.parser")
    filings = []
    seen = set()

    for a in soup.find_all("a", href=True):
        href  = a["href"]
        title = a.get_text(separator=" ", strip=True)

        # Must mention the year somewhere
        if str(year) not in title and str(year) not in href:
            continue
        # Skip empty, anchor-only, or JS links
        if not title or href.startswith("#") or href.startswith("javascript"):
            continue
        if href in seen:
            continue
        seen.add(href)

        # Resolve relative URLs
        if href.startswith("/"):
            from urllib.parse import urlparse
            base = urlparse(page_url)
            href = f"{base.scheme}://{base.netloc}{href}"
        elif not href.startswith("http"):
            continue

        filings.append(Filing(
            company=company,
            form_type="press_release",
            year=year,
            title=title[:120],
            url=href,
            source=source,
        ))

    log.info("%s: found %d link(s) on %s for %d", source, len(filings), page_url, year)
    return filings


# ---------------------------------------------------------------------------
# Newsroom adapter — all companies
# ---------------------------------------------------------------------------

class TeslaNewsroomAdapter:
    """
    Scrapes Tesla's newsroom (press releases) and IR page (SEC filings, earnings).
    Newsroom: https://www.tesla.com/en_US/blog
    IR:       https://ir.tesla.com/press-releases
    """
    name = "Tesla_Newsroom"
    SOURCES = [
        "https://www.tesla.com/en_US/blog",
        "https://ir.tesla.com/press-releases",
    ]

    def get_filings(self, company: str, year: int) -> list[Filing]:
        results = []
        for url in self.SOURCES:
            results += _scrape_links(company, year, url, source=self.name)
        return results


class DisneyNewsroomAdapter:
    """
    Scrapes Disney's press room and investor relations news.
    Newsroom: https://press.disneyplus.com  (Disney+)
              https://thewaltdisneycompany.com/news
    IR:       https://thewaltdisneycompany.com/investor-relations
    """
    name = "Disney_Newsroom"
    SOURCES = [
        "https://thewaltdisneycompany.com/news",
        "https://thewaltdisneycompany.com/investor-relations",
    ]

    def get_filings(self, company: str, year: int) -> list[Filing]:
        results = []
        for url in self.SOURCES:
            results += _scrape_links(company, year, url, source=self.name)
        return results


class NetflixNewsroomAdapter:
    """
    Scrapes Netflix's press room and IR page.
    Newsroom: https://media.netflix.com/en/press-releases
    IR:       https://ir.netflix.net/ir-overview/press-releases
    """
    name = "Netflix_Newsroom"
    SOURCES = [
        "https://media.netflix.com/en/press-releases",
        "https://ir.netflix.net/ir-overview/press-releases",
    ]

    def get_filings(self, company: str, year: int) -> list[Filing]:
        results = []
        for url in self.SOURCES:
            results += _scrape_links(company, year, url, source=self.name)
        return results


class VolkswagenNewsroomAdapter:
    """
    Scrapes VW's newsroom and investor relations page.
    Newsroom: https://www.volkswagen-newsroom.com/en/press-releases
    IR:       https://www.volkswagen-group.com/en/press-releases
    """
    name = "VW_Newsroom"
    SOURCES = [
        "https://www.volkswagen-newsroom.com/en/press-releases",
        "https://www.volkswagen-group.com/en/press-releases",
    ]

    def get_filings(self, company: str, year: int) -> list[Filing]:
        results = []
        for url in self.SOURCES:
            results += _scrape_links(company, year, url, source=self.name)
        return results

# ---------------------------------------------------------------------------
# Downloader
# ---------------------------------------------------------------------------

class Downloader:
    """Downloads SEC HTML docs, VW PDFs, or saves Bundesanzeiger text."""

    def __init__(self):
        self.session = requests.Session()
        self.session.headers.update({
            "User-Agent": SEC_USER_AGENT,
            "Accept":     "text/html,application/pdf,*/*",
        })

    def save(self, filing: Filing, dest_dir: Path) -> Optional[Path]:
        dest_dir.mkdir(parents=True, exist_ok=True)
        safe = re.sub(r"[^\w\-]+", "_", filing.title)[:80]

        # Bundesanzeiger: content is already in memory as text
        if filing.source == "Bundesanzeiger":
            if not filing._content:
                return None
            dest = dest_dir / f"{safe}.txt"
            if not dest.exists():
                dest.write_text(filing._content, encoding="utf-8")
                log.info("    saved: %s", dest.name)
            else:
                log.info("    skip (exists): %s", dest.name)
            return dest

        # SEC / VW IR: download from URL
        time.sleep(REQUEST_DELAY)
        # VW IR pages need a browser UA
        if filing.source == "VW_IR":
            self.session.headers["User-Agent"] = BROWSER_UA

        try:
            r = self.session.get(
                filing.url, timeout=30, stream=True, allow_redirects=True
            )
            r.raise_for_status()
        except requests.RequestException as exc:
            log.warning("    download failed '%s': %s", filing.url, exc)
            return None

        ct  = r.headers.get("content-type", "")
        ext = ".pdf" if ("pdf" in ct or filing.url.lower().endswith(".pdf")) else ".htm"
        dest = dest_dir / f"{safe}{ext}"

        if dest.exists():
            log.info("    skip (exists): %s", dest.name)
        else:
            with open(dest, "wb") as f:
                for chunk in r.iter_content(8192):
                    f.write(chunk)
            log.info("    saved: %s", dest.name)

        return dest


# ---------------------------------------------------------------------------
# Bulk runner
# ---------------------------------------------------------------------------

def run_all(
    companies: list[str] = list(COMPANIES.keys()),
    years: list[int] = YEARS,
) -> None:
    adapters = {
        "Tesla":      TeslaNewsroomAdapter(),
        "Disney":     DisneyNewsroomAdapter(),
        "Netflix":    NetflixNewsroomAdapter(),
        "Volkswagen": VolkswagenNewsroomAdapter(),
    }
    dl = Downloader()
    total = 0
    for company in companies:
        for year in years:
            out_dir = OUTPUT_DIR / company / str(year)
            filings = adapters[company].get_filings(company, year)
            saved = []
            for filing in filings:
                path = dl.save(filing, out_dir)
                if path:
                    saved.append(path)
            n = len(saved)
            total += n
            print(f"\n✓  {n} file(s) for '{company}' ({year}) → {out_dir}")
    print(f"\n{'='*55}\nDone — {total} file(s) saved in total.")


# ---------------------------------------------------------------------------
# Entry point
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    run_all()

12:14:09  WARNING   Tesla_Newsroom: fetch failed for https://www.tesla.com/en_US/blog: 403 Client Error: Forbidden for url: https://www.tesla.com/en_US/blog
12:14:14  WARNING   Tesla_Newsroom: fetch failed for https://ir.tesla.com/press-releases: 403 Client Error: Forbidden for url: https://ir.tesla.com/press-releases



✓  0 file(s) for 'Tesla' (2020) → newsroom_reports\Tesla\2020


12:14:18  WARNING   Tesla_Newsroom: fetch failed for https://www.tesla.com/en_US/blog: 403 Client Error: Forbidden for url: https://www.tesla.com/en_US/blog
12:14:22  WARNING   Tesla_Newsroom: fetch failed for https://ir.tesla.com/press-releases: 403 Client Error: Forbidden for url: https://ir.tesla.com/press-releases



✓  0 file(s) for 'Tesla' (2021) → newsroom_reports\Tesla\2021


12:14:26  WARNING   Tesla_Newsroom: fetch failed for https://www.tesla.com/en_US/blog: 403 Client Error: Forbidden for url: https://www.tesla.com/en_US/blog
12:14:30  WARNING   Tesla_Newsroom: fetch failed for https://ir.tesla.com/press-releases: 403 Client Error: Forbidden for url: https://ir.tesla.com/press-releases


...


✓  2 file(s) for 'Volkswagen' (2024) → newsroom_reports\Volkswagen\2024


12:19:59  INFO      VW_Newsroom: found 3 link(s) on https://www.volkswagen-newsroom.com/en/press-releases for 2025
12:20:03  INFO      VW_Newsroom: found 1 link(s) on https://www.volkswagen-group.com/en/press-releases for 2025
12:20:07  INFO          saved: IAA_MOBILITY_2025_-_The_Volkswagen_News.htm
12:20:11  INFO          saved: 50_years_of_Polo_2025_.htm
12:20:18  INFO          skip (exists): IAA_MOBILITY_2025_-_The_Volkswagen_News.htm
12:20:24  INFO          saved: 2025_Year-end_Reporting_of_the_Volkswagen_Group_Companies.htm



✓  4 file(s) for 'Volkswagen' (2025) → newsroom_reports\Volkswagen\2025


12:20:28  INFO      VW_Newsroom: found 3 link(s) on https://www.volkswagen-newsroom.com/en/press-releases for 2026
12:20:36  INFO      VW_Newsroom: found 11 link(s) on https://www.volkswagen-group.com/en/press-releases for 2026
12:20:41  INFO          saved: 50_years_of_GTI_2026_.htm
12:20:46  INFO          saved: Annual_Media_Call_2026.htm
12:20:49  INFO          skip (exists): Annual_Media_Call_2026.htm
12:20:52  INFO          saved: Annual_General_Meeting.htm
12:20:57  INFO          saved: 06_25_2026_Press_Release_Volkswagen_Group_Award_2026_recognizes_outstanding_supp.htm
12:21:00  INFO          saved: 06_24_2026_Press_Release_Key_step_towards_streamlining_investment_portfolio_Volk.htm
12:21:04  INFO          saved: 06_23_2026_Press_Release_Volkswagen_Group_and_Elli_bring_Vehicle-to-Grid_offer_t.htm
12:21:08  INFO          saved: 06_18_2026_Press_Release_Volkswagen_shareholders_formally_approve_Board_of_Manag.htm
12:21:11  INFO          saved: 06_18_2026_Press_Release_Strengthen_su


✓  14 file(s) for 'Volkswagen' (2026) → newsroom_reports\Volkswagen\2026

Done — 27 file(s) saved in total.
